In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.5055000000000002, 10: 0.6234999999999999, 20: 0.6549999999999999, 30: 0.6875000000000001, 40: 0.7054999999999998, 50: 0.7195000000000003, 60: 0.7205000000000001, 70: 0.7240000000000002, 80: 0.7345, 90: 0.7380000000000001, 100: 0.7394999999999999, 110: 0.7500000000000001, 120: 0.7524999999999998, 130: 0.7585, 140: 0.752, 150: 0.7605000000000002, 160: 0.7595000000000002, 170: 0.7689999999999999, 180: 0.7765, 190: 0.7755, 200: 0.7745000000000001, 210: 0.7789999999999998, 220: 0.7740000000000002, 230: 0.7715000000000001, 240: 0.7829999999999998, 250: 0.7855000000000001, 260: 0.7789999999999999, 270: 0.7825000000000001, 280: 0.7845000000000001, 290: 0.7907692307692307, 300: 0.78}
{0: 0.00943975, 10: 0.00521775, 20: 0.0032749999999999993, 30: 0.004913750000000001, 40: 0.00461975, 50: 0.00348975, 60: 0.0032897499999999993, 70: 0.0026839999999999998, 80: 0.0037997499999999997, 90: 0.004475999999999999, 100: 0.00450975, 110: 0.00432, 120: 0.00313375, 130: 0.004007749999999999, 140: 0.0023